# RAG - Chatbot Tim Legal berbasis Dokumen UU/PP (PGABL)
**Nama:** Stanley Nathanael Wijaya

Notebook ini membungkus model hasil fine-tuning (`Fine-tuning_submission_PGABL_...ipynb`) dengan
pipeline **Retrieval-Augmented Generation (RAG)** di atas 4 dokumen hukum resmi:

- PP Nomor 5 Tahun 2021
- PP Nomor 35 Tahun 2021
- PP Nomor 51 Tahun 2023
- UU Nomor 6 Tahun 2023

**Rekomendasi environment:** Google Colab / Kaggle dengan GPU (T4 ke atas).

Struktur notebook:
1. Instalasi dependency & unduh 4 dokumen PDF
2. Load PDF + text splitting (chunk size & overlap eksplisit)
3. Embedding open-source -> Vector DB lokal (ChromaDB)
4. Load model hasil fine-tuning + prompt `{context}`/`{question}`
5. Interface sederhana (Python loop / Gradio)
6. **Skilled:** metadata enrichment & filtering + sitasi, Ensemble Retriever (BM25 + vektor), Parent-Child Retriever
7. **Advanced:** HyDE, Reranker (Cross-Encoder) + fallback DuckDuckGo Search


## 1. Instalasi Dependency & Unduh Dokumen

In [ ]:
%%capture
!pip install -q langchain langchain-community langchain-huggingface chromadb pypdf \
    sentence-transformers rank_bm25 gradio duckduckgo-search gdown unsloth

In [ ]:
import os

DATA_DIR = "knowledge_rag"
os.makedirs(DATA_DIR, exist_ok=True)

# Folder Google Drive resmi berisi 4 dokumen UU/PP wajib (lihat halaman kriteria submission)
DRIVE_FOLDER_URL = "https://drive.google.com/drive/folders/1LHZ1IncPmmUN5kytFu3i7MoaafFrKDql"

if not any(f.endswith(".pdf") for f in os.listdir(DATA_DIR)):
    import gdown

    gdown.download_folder(DRIVE_FOLDER_URL, output=DATA_DIR, quiet=False, use_cookies=False)

pdf_paths = [
    os.path.join(root, f)
    for root, _, files in os.walk(DATA_DIR)
    for f in files
    if f.lower().endswith(".pdf")
]
print(f"{len(pdf_paths)} dokumen PDF ditemukan:")
for p in pdf_paths:
    print(" -", p)
assert len(pdf_paths) == 4, "Seluruh 4 dokumen UU/PP WAJIB digunakan."

## 2. Load Dokumen PDF & Text Splitting

Ukuran `chunk_size` dan `chunk_overlap` ditentukan **secara eksplisit** (1000 karakter / 150
karakter overlap) agar potongan cukup besar untuk memuat konteks pasal, tapi tetap ringkas untuk
retrieval yang presisi.


In [ ]:
from langchain_community.document_loaders import PyPDFLoader

raw_documents = []
for path in pdf_paths:
    loader = PyPDFLoader(path)
    docs = loader.load()
    raw_documents.extend(docs)

print(f"Total {len(raw_documents)} halaman dimuat dari {len(pdf_paths)} dokumen.")

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

CHUNK_SIZE = 1000
CHUNK_OVERLAP = 150

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\n\n", "\n", ". ", " ", ""],
)

chunks = text_splitter.split_documents(raw_documents)
print(f"Total {len(chunks)} chunk dihasilkan (chunk_size={CHUNK_SIZE}, overlap={CHUNK_OVERLAP}).")
print(chunks[0].page_content[:300])

### Metadata Enrichment (Skilled)

Menambahkan metadata eksplisit per chunk: nama dokumen, nomor peraturan, tipe peraturan (UU/PP),
dan nomor halaman — dipakai untuk metadata filtering & sitasi jawaban.


In [ ]:
import re


def enrich_metadata(chunk):
    filename = os.path.basename(chunk.metadata.get("source", "unknown.pdf"))
    doc_type = "UU" if filename.upper().startswith("UU") else "PP"
    match = re.search(r"Nomor\s+(\d+)\s+Tahun\s+(\d{4})", filename, re.IGNORECASE)
    nomor, tahun = (match.group(1), match.group(2)) if match else ("?", "?")

    chunk.metadata.update(
        {
            "document_name": filename,
            "doc_type": doc_type,
            "nomor_peraturan": nomor,
            "tahun_peraturan": tahun,
            "citation": f"{doc_type} No. {nomor}/{tahun}, hal. {chunk.metadata.get('page', 0) + 1}",
        }
    )
    return chunk


chunks = [enrich_metadata(c) for c in chunks]
chunks[0].metadata

## 3. Embedding Open-Source -> Vector Database Lokal (ChromaDB)

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

EMBEDDING_MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
embeddings = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL)

from langchain_community.vectorstores import Chroma

PERSIST_DIR = "chroma_legal_db"
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=PERSIST_DIR,
    collection_name="legal_docs",
)
print(f"Vector DB berisi {vectorstore._collection.count()} chunk.")

## 4. Load Model Hasil Fine-tuning

In [ ]:
import os
from getpass import getpass

from huggingface_hub import login

HF_TOKEN = os.environ.get("HF_TOKEN") or getpass("Masukkan Hugging Face Token: ")
login(token=HF_TOKEN)

HF_USERNAME = os.environ.get("HF_USERNAME") or input("Masukkan username Hugging Face kamu: ")
# Ganti ke *-grpo jika ingin memakai model hasil GRPO (Advanced)
FT_REPO_ID = os.environ.get("FT_REPO_ID") or f"{HF_USERNAME}/qwen2.5-1.5b-legal-chatbot-id"
print("Memuat model dari:", FT_REPO_ID)

In [ ]:
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template

max_seq_length = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=FT_REPO_ID,
    max_seq_length=max_seq_length,
    dtype=None,
    load_in_4bit=True,
)
tokenizer = get_chat_template(tokenizer, chat_template="chatml")
FastLanguageModel.for_inference(model)

RAG_SYSTEM_PROMPT = (
    "Kamu adalah asisten AI internal Tim Legal perusahaan. Jawablah HANYA berdasarkan "
    "konteks dokumen resmi yang diberikan. Jika jawaban tidak ada dalam konteks, katakan "
    "dengan jujur bahwa informasi tidak ditemukan pada dokumen. Jawab dalam Bahasa Indonesia."
)

RAG_PROMPT_TEMPLATE = """Konteks dokumen:
{context}

Pertanyaan: {question}

Jawablah pertanyaan di atas HANYA berdasarkan konteks di atas."""


def generate_answer(question, context, max_new_tokens=400):
    prompt = RAG_PROMPT_TEMPLATE.format(context=context, question=question)
    convo = [
        {"role": "system", "content": RAG_SYSTEM_PROMPT},
        {"role": "user", "content": prompt},
    ]
    inputs = tokenizer.apply_chat_template(
        convo, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to(model.device)
    outputs = model.generate(
        input_ids=inputs, max_new_tokens=max_new_tokens, temperature=0.3, do_sample=True
    )
    return tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)

## 5. Retriever

**Basic:** similarity search sederhana dari ChromaDB.

**Skilled:** metadata filtering, Ensemble Retriever (BM25 keyword + vektor semantik, bobot
ditentukan eksplisit, ambil >= 5 dokumen), dan Parent-Child Retriever (child chunk kecil untuk
pencarian, parent chunk besar/halaman utuh untuk konteks LLM).


In [ ]:
# --- Basic retriever ---
basic_retriever = vectorstore.as_retriever(search_kwargs={"k": 5})


def retrieve_basic(question):
    return basic_retriever.invoke(question)

In [ ]:
# --- Skilled: metadata filtering ---
def retrieve_with_filter(question, doc_type=None, nomor_peraturan=None, k=5):
    where = {}
    if doc_type:
        where["doc_type"] = doc_type
    if nomor_peraturan:
        where["nomor_peraturan"] = nomor_peraturan
    filtered_retriever = vectorstore.as_retriever(
        search_kwargs={"k": k, "filter": where} if where else {"k": k}
    )
    return filtered_retriever.invoke(question)

In [ ]:
# --- Skilled: Ensemble Retriever (BM25 keyword + vektor semantik), bobot eksplisit ---
from langchain.retrievers import EnsembleRetriever
from langchain_community.retrievers import BM25Retriever

bm25_retriever = BM25Retriever.from_documents(chunks)
bm25_retriever.k = 5

vector_retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, vector_retriever],
    weights=[0.4, 0.6],  # 40% keyword (BM25), 60% semantik (vektor)
)

sample_docs = ensemble_retriever.invoke("upah lembur pekerja")
print(f"Ensemble retriever mengambil {len(sample_docs)} dokumen.")

In [ ]:
# --- Skilled: Parent-Child Retriever ---
from langchain.retrievers import ParentDocumentRetriever
from langchain.storage import InMemoryStore

parent_splitter = RecursiveCharacterTextSplitter(chunk_size=3000, chunk_overlap=200)
child_splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=50)

parent_child_vectorstore = Chroma(
    collection_name="legal_docs_parent_child",
    embedding_function=embeddings,
    persist_directory="chroma_legal_db_parent_child",
)
parent_docstore = InMemoryStore()

parent_child_retriever = ParentDocumentRetriever(
    vectorstore=parent_child_vectorstore,
    docstore=parent_docstore,
    child_splitter=child_splitter,
    parent_splitter=parent_splitter,
)
parent_child_retriever.add_documents(raw_documents)
print("Parent-Child retriever siap. Jumlah parent chunk:", len(list(parent_docstore.yield_keys())))

## 6. Sitasi Jawaban (Skilled)

Menyusun konteks dari chunk terpilih sekaligus daftar sitasi (`document_name`, halaman) yang
dilampirkan di akhir jawaban.


In [ ]:
def build_context_and_citations(docs):
    context_parts = []
    citations = []
    for d in docs:
        context_parts.append(d.page_content)
        citation = d.metadata.get("citation", d.metadata.get("source", "unknown"))
        if citation not in citations:
            citations.append(citation)
    return "\n\n---\n\n".join(context_parts), citations


def answer_with_citation(question, retriever_fn=retrieve_basic, **retriever_kwargs):
    docs = retriever_fn(question, **retriever_kwargs) if retriever_kwargs else retriever_fn(question)
    context, citations = build_context_and_citations(docs)
    answer = generate_answer(question, context)
    citation_text = "\n".join(f"- {c}" for c in citations)
    return f"{answer}\n\n**Sumber:**\n{citation_text}"

## 7. HyDE + Reranker + Fallback DuckDuckGo (Advanced)

1. **HyDE**: LLM membuat >= 2 jawaban hipotetis (halusinasi awal) dari pertanyaan, lalu embedding
   dari jawaban-jawaban tersebut (bukan pertanyaan mentah) dipakai untuk retrieval.
2. **Reranker**: Cross-Encoder mengurutkan ulang hasil retrieval dan hanya mengambil Top-K (K=3).
3. Jika **Relevance Score Top-1** dari reranker berada di bawah threshold, sistem beralih ke
   pencarian internet (DuckDuckGo) sebagai fallback.


In [ ]:
def generate_hyde_documents(question, n=2):
    hypothetical_answers = []
    for _ in range(n):
        convo = [
            {"role": "system", "content": "Jawablah pertanyaan hukum berikut secara singkat, walau tidak yakin."},
            {"role": "user", "content": question},
        ]
        inputs = tokenizer.apply_chat_template(
            convo, tokenize=True, add_generation_prompt=True, return_tensors="pt"
        ).to(model.device)
        outputs = model.generate(input_ids=inputs, max_new_tokens=150, temperature=0.9, do_sample=True)
        hypothetical_answers.append(tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True))
    return hypothetical_answers


def retrieve_with_hyde(question, k=8):
    hyde_docs = generate_hyde_documents(question, n=2)
    combined_query = question + "\n" + "\n".join(hyde_docs)
    hyde_embedding = embeddings.embed_query(combined_query)
    return vectorstore.similarity_search_by_vector(hyde_embedding, k=k)

In [ ]:
from sentence_transformers import CrossEncoder

RERANKER_MODEL = "cross-encoder/ms-marco-MiniLM-L-6-v2"
reranker = CrossEncoder(RERANKER_MODEL)

RELEVANCE_THRESHOLD = 0.0  # skor logit cross-encoder; sesuaikan berdasarkan observasi empiris

def rerank(question, docs, top_k=3):
    pairs = [(question, d.page_content) for d in docs]
    scores = reranker.predict(pairs)
    ranked = sorted(zip(docs, scores), key=lambda x: x[1], reverse=True)
    top1_score = float(ranked[0][1]) if ranked else -999
    return [d for d, _ in ranked[:top_k]], top1_score

In [ ]:
from duckduckgo_search import DDGS


def web_search_fallback(question, max_results=3):
    with DDGS() as ddgs:
        results = list(ddgs.text(question, max_results=max_results))
    return "\n\n".join(f"{r['title']}: {r['body']}" for r in results)


def advanced_rag_answer(question):
    candidate_docs = retrieve_with_hyde(question, k=8)
    top_docs, top1_score = rerank(question, candidate_docs, top_k=3)
    print(f"[debug] Reranker Top-1 relevance score: {top1_score:.4f}")

    if top1_score < RELEVANCE_THRESHOLD:
        print("[debug] Skor di bawah threshold -> fallback ke DuckDuckGo Search")
        context = web_search_fallback(question)
        citations = ["DuckDuckGo Search (internet)"]
    else:
        context, citations = build_context_and_citations(top_docs)

    answer = generate_answer(question, context)
    citation_text = "\n".join(f"- {c}" for c in citations)
    return f"{answer}\n\n**Sumber:**\n{citation_text}"

## 8. Interface Sederhana

### Opsi A - Interactive Python Loop

In [ ]:
from IPython.display import Markdown, display


def run_chat_loop(use_advanced=False):
    print("Ketik 'exit' untuk keluar.")
    while True:
        question = input("\nPertanyaan Anda: ")
        if question.strip().lower() == "exit":
            break
        if use_advanced:
            response = advanced_rag_answer(question)
        else:
            response = answer_with_citation(question, retriever_fn=lambda q: ensemble_retriever.invoke(q))
        display(Markdown(response))


# Jalankan salah satu baris berikut secara interaktif:
# run_chat_loop(use_advanced=False)
# run_chat_loop(use_advanced=True)

### Opsi B - Gradio Interface

In [ ]:
import gradio as gr


def gradio_answer(question, use_advanced):
    if use_advanced:
        return advanced_rag_answer(question)
    return answer_with_citation(question, retriever_fn=lambda q: ensemble_retriever.invoke(q))


demo = gr.Interface(
    fn=gradio_answer,
    inputs=[
        gr.Textbox(label="Pertanyaan", placeholder="Tanyakan seputar UU/PP Ketenagakerjaan..."),
        gr.Checkbox(label="Gunakan pipeline Advanced (HyDE + Reranker + DuckDuckGo fallback)", value=False),
    ],
    outputs=gr.Markdown(label="Jawaban"),
    title="Chatbot Tim Legal berbasis RAG",
    description="Fine-tuned SLM + RAG di atas dokumen UU/PP Ketenagakerjaan.",
)

# demo.launch(share=True)  # uncomment untuk menjalankan UI interaktif

## 9. Uji Coba Test Case Wajib (Advanced)

Prompt: *"Saya staf admin, kemarin lembur 3 jam untuk beresin laporan. Apakah saya berhak dapat uang
lembur?"*


In [ ]:
test_question = (
    "Saya staf admin, kemarin lembur 3 jam untuk beresin laporan. "
    "Apakah saya berhak dapat uang lembur?"
)

# Gunakan model hasil GRPO (FT_REPO_ID diarahkan ke *-grpo) agar proses <think> muncul di jawaban.
response = advanced_rag_answer(test_question)
display(Markdown(response))